# MCP Integration for Retrieval Services
Companion notebook for the **MCP Integration for Retrieval Services** section of Chapter 10.

Use **tools** for actions like search -- a callable function with a typed input schema and a structured result. Use **resources** for governed, read-only context access -- a client calls a search tool first, then reads a resource to get the exact content with metadata. Keep the boundary clean: tools return IDs/scores/URIs, resources return the actual text.

The book's tool schema for `search_docs`:

In [1]:
search_docs_schema = {
    "name": "search_docs",
    "description": "Search approved documents with filters",
    "inputSchema": {
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "top_k": {"type": "integer", "default": 5},
            "tenant_id": {"type": "string"},
            "doc_type": {"type": "string"},
        },
        "required": ["query", "tenant_id"],
    },
}
import json
print(json.dumps(search_docs_schema, indent=2))

{
  "name": "search_docs",
  "description": "Search approved documents with filters",
  "inputSchema": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string"
      },
      "top_k": {
        "type": "integer",
        "default": 5
      },
      "tenant_id": {
        "type": "string"
      },
      "doc_type": {
        "type": "string"
      }
    },
    "required": [
      "query",
      "tenant_id"
    ]
  }
}


## A small MCP server exposing that tool -- plus a resource
`search_docs` is a **tool**: it takes a query and tenant, returns IDs, titles, scores, and resource URIs -- never the raw text. `resource://kb/{doc_id}` is a **resource**: it returns the actual approved content, read-only, no side effects. Both enforce the tenant check.

In [2]:
%%writefile retrieval_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("RetrievalServer")

_DOCS = [
    {"doc_id": "policy_2025_refunds", "title": "Refund Policy", "tenant_id": "acme",
     "text": "Refunds are allowed within 30 days of purchase for annual plans."},
    {"doc_id": "billing_faq", "title": "Billing FAQ", "tenant_id": "acme",
     "text": "Billing questions are answered by the support team within one business day."},
    {"doc_id": "globex_refunds", "title": "Globex Refund Policy", "tenant_id": "globex",
     "text": "Globex refunds follow a 15 day window."},
]


@mcp.tool()
def search_docs(query: str, tenant_id: str, top_k: int = 5) -> dict:
    """Search approved documents with filters."""
    results = retriever_search(query=query, filters={"tenant_id": tenant_id}, top_k=top_k)
    return {
        "results": [
            {"doc_id": r["doc_id"], "title": r["title"], "score": r["score"], "uri": f"resource://kb/{r['doc_id']}"}
            for r in results
        ]
    }


def retriever_search(query: str, filters: dict, top_k: int) -> list[dict]:
    """Stand-in for a real retriever -- simple prefix overlap so 'refund' still credits 'refunds'."""
    q_words = query.lower().split()
    scored = []
    for doc in _DOCS:
        if doc["tenant_id"] != filters["tenant_id"]:
            continue
        text_words = doc["text"].lower().split()
        overlap = sum(1 for qw in q_words if any(tw.startswith(qw) or qw.startswith(tw) for tw in text_words))
        scored.append({**doc, "score": overlap})
    scored.sort(key=lambda r: r["score"], reverse=True)
    return scored[:top_k]


@mcp.resource("resource://kb/{doc_id}")
def get_document(doc_id: str) -> dict:
    """Read the approved text and metadata for one document."""
    for doc in _DOCS:
        if doc["doc_id"] == doc_id:
            return doc
    return {"doc_id": doc_id, "error": "not found"}


if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing retrieval_server.py


In [3]:
import sys, os

# Jupyter's stdout/stderr streams don't expose a real OS file descriptor, which crashes
# the MCP SDK's Windows subprocess-spawning fallback (it isn't needed outside Jupyter).
if os.name == "nt":
    try:
        sys.stderr.fileno()
    except Exception:
        sys.stderr = os.fdopen(os.dup(2), "w")

import asyncio
import json
import sys
from pathlib import Path
from typing import Any

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def call_tool(session: ClientSession, name: str, arguments: dict) -> Any:
    """Call an MCP tool and unwrap its result into a plain Python value."""
    result = await session.call_tool(name, arguments)
    if result.structuredContent is not None:
        content = result.structuredContent
        return content.get("result", content) if isinstance(content, dict) else content
    text = result.content[0].text
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return text


async def read_resource(session: ClientSession, uri: str) -> Any:
    """Read an MCP resource and unwrap it into a plain Python value."""
    result = await session.read_resource(uri)
    text = result.contents[0].text
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return text

In [4]:
server_path = Path.cwd() / "retrieval_server.py"
params = StdioServerParameters(command=sys.executable, args=[str(server_path)])

async with stdio_client(params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        hits = await call_tool(session, "search_docs", {"query": "refund window", "tenant_id": "acme"})
        print("tool result (IDs, scores, URIs -- no raw text yet):")
        for r in hits["results"]:
            print(" ", r)

        top_uri = hits["results"][0]["uri"]
        doc = await read_resource(session, top_uri)
        print("\nresource result (actual approved content):")
        print(" ", doc)

tool result (IDs, scores, URIs -- no raw text yet):
  {'doc_id': 'policy_2025_refunds', 'title': 'Refund Policy', 'score': 1, 'uri': 'resource://kb/policy_2025_refunds'}
  {'doc_id': 'billing_faq', 'title': 'Billing FAQ', 'score': 0, 'uri': 'resource://kb/billing_faq'}

resource result (actual approved content):
  {'doc_id': 'policy_2025_refunds', 'title': 'Refund Policy', 'tenant_id': 'acme', 'text': 'Refunds are allowed within 30 days of purchase for annual plans.'}


This separation is what makes retrieval reusable: a support bot, an analyst agent, and an internal assistant can all call the same MCP server, while each keeps its own LangGraph workflow logic on the client side (notebook 08).